In [1]:
# HealthGuard AI - Hypertension Data Cleaning
# Author: HealthGuard AI Team
# Date: 2026

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
%matplotlib inline

plt.style.use('seaborn-v0_8')

print("=" * 50)
print("  HealthGuard AI - Hypertension Cleaning")
print("=" * 50)
print("Libraries Loaded Successfully!")

  HealthGuard AI - Hypertension Cleaning
Libraries Loaded Successfully!


In [2]:
# Load Hypertension Dataset

df = pd.read_csv(
    "E:/HealthGuard_AI/data/raw/hypertension_dataset.csv")

print(f"Original Shape: {df.shape}")
print(f"\nColumn Types:")
print(df.dtypes)
df.head()

Original Shape: (1985, 11)

Column Types:
Age                   int64
Salt_Intake         float64
Stress_Score          int64
BP_History           object
Sleep_Duration      float64
BMI                 float64
Medication           object
Family_History       object
Exercise_Level       object
Smoking_Status       object
Has_Hypertension     object
dtype: object


,Age,Salt_Intake,Stress_Score,BP_History,Sleep_Duration,BMI,Medication,Family_History,Exercise_Level,Smoking_Status,Has_Hypertension
0,69,8.0,9,Normal,6.4,25.8,NaN,Yes,Low,Non-Smoker,Yes
1,32,11.7,10,Normal,5.4,23.4,NaN,No,Low,Non-Smoker,No
2,78,9.5,3,Normal,7.1,18.7,NaN,No,Moderate,Non-Smoker,No
3,38,10.0,10,Hypertension,4.2,22.1,ACE Inhibitor,No,Low,Non-Smoker,Yes
4,41,9.8,1,Prehypertension,5.8,16.2,Other,No,Moderate,Non-Smoker,No


In [3]:
# Step 1: Remove Duplicates

print("=" * 50)
print("STEP 1: REMOVE DUPLICATES")
print("=" * 50)

before = len(df)
df = df.drop_duplicates()
after = len(df)

print(f"Before: {before} rows")
print(f"After: {after} rows")
print(f"Removed: {before - after} duplicates")

STEP 1: REMOVE DUPLICATES
Before: 1985 rows
After: 1985 rows
Removed: 0 duplicates


In [4]:
# Step 2: Handle Missing Values

print("=" * 50)
print("STEP 2: HANDLE MISSING VALUES")
print("=" * 50)

print("Missing Values:")
print(df.isnull().sum())

# Numeric - median
numeric_cols = df.select_dtypes(
    include=['float64', 'int64']).columns
df[numeric_cols] = df[numeric_cols].fillna(
    df[numeric_cols].median())

# Categorical - mode
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("\nAfter Fix:")
print(df.isnull().sum())

STEP 2: HANDLE MISSING VALUES
Missing Values:
Age                   0
Salt_Intake           0
Stress_Score          0
BP_History            0
Sleep_Duration        0
BMI                   0
Medication          799
Family_History        0
Exercise_Level        0
Smoking_Status        0
Has_Hypertension      0
dtype: int64

After Fix:
Age                 0
Salt_Intake         0
Stress_Score        0
BP_History          0
Sleep_Duration      0
BMI                 0
Medication          0
Family_History      0
Exercise_Level      0
Smoking_Status      0
Has_Hypertension    0
dtype: int64


In [5]:
# Step 3: Encode Target Column
# Has_Hypertension: Yes=1, No=0

print("=" * 50)
print("STEP 3: ENCODE TARGET")
print("=" * 50)

df['Has_Hypertension'] = df['Has_Hypertension'].map(
    {'Yes': 1, 'No': 0})

print("Has_Hypertension Encoded:")
print(df['Has_Hypertension'].value_counts())

STEP 3: ENCODE TARGET
Has_Hypertension Encoded:
Has_Hypertension
1    1032
0     953
Name: count, dtype: int64


In [6]:
# Step 4: Encode Categorical Features
# Using Label Encoding

print("=" * 50)
print("STEP 4: ENCODE CATEGORICAL FEATURES")
print("=" * 50)

le = LabelEncoder()

cat_cols = df.select_dtypes(include=['object']).columns
print("Encoding columns:")
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))
    print(f" {col} encoded")

STEP 4: ENCODE CATEGORICAL FEATURES
Encoding columns:
 BP_History encoded
 Medication encoded
 Family_History encoded
 Exercise_Level encoded
 Smoking_Status encoded


In [7]:
# Step 5: Remove Outliers

print("=" * 50)
print("STEP 5: REMOVE OUTLIERS (IQR METHOD)")
print("=" * 50)

before = len(df)

outlier_cols = ['Age', 'Salt_Intake',
                'Stress_Score', 'Sleep_Duration', 'BMI']

for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {outliers} outliers removed")

    df = df[(df[col] >= lower) & (df[col] <= upper)]

after = len(df)
print(f"\nBefore: {before} rows")
print(f"After: {after} rows")

STEP 5: REMOVE OUTLIERS (IQR METHOD)
Age: 0 outliers removed
Salt_Intake: 17 outliers removed
Stress_Score: 0 outliers removed
Sleep_Duration: 12 outliers removed
BMI: 15 outliers removed

Before: 1985 rows
After: 1941 rows


In [8]:
# Step 6: Feature Scaling

print("=" * 50)
print("STEP 6: FEATURE SCALING")
print("=" * 50)

X = df.drop('Has_Hypertension', axis=1)
y = df['Has_Hypertension']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("Scaling Complete!")
print(f"Features Shape: {X_scaled.shape}")

STEP 6: FEATURE SCALING
Scaling Complete!
Features Shape: (1941, 10)


In [9]:
# Train Test Split FIRST, then SMOTE only on Training data

print("=" * 50)
print("TRAIN TEST SPLIT (BEFORE SMOTE)")
print("=" * 50)

X_train_raw, X_test, y_train_raw, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y)

print(f"Train (before SMOTE): {len(X_train_raw)}")
print(f"Test (untouched): {len(X_test)}")

print("\n" + "=" * 50)
print("SMOTE ON TRAINING DATA ONLY")
print("=" * 50)

print("Before SMOTE:")
print(f"No Hypertension (0): {(y_train_raw==0).sum()}")
print(f"Hypertension (1): {(y_train_raw==1).sum()}")

smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train_raw, y_train_raw)

print("\nAfter SMOTE:")
print(f"No Hypertension (0): {(y_train==0).sum()}")
print(f"Hypertension (1): {(y_train==1).sum()}")
print(f"Final Training: {len(X_train)}, Final Testing: {len(X_test)}")

TRAIN TEST SPLIT (BEFORE SMOTE)
Train (before SMOTE): 1552
Test (untouched): 389

SMOTE ON TRAINING DATA ONLY
Before SMOTE:
No Hypertension (0): 747
Hypertension (1): 805

After SMOTE:
No Hypertension (0): 805
Hypertension (1): 805
Final Training: 1610, Final Testing: 389


In [10]:
# Step 9: Save Cleaned Data

print("=" * 50)
print("STEP 9: SAVE CLEANED DATA")
print("=" * 50)

df_cleaned = pd.concat([X_scaled,
                        y.reset_index(drop=True)], axis=1)
df_cleaned.to_csv(
    "E:/HealthGuard_AI/data/processed/hypertension_cleaned_final.csv",
    index=False)

X_train.to_csv(
    "E:/HealthGuard_AI/data/processed/hypertension_X_train.csv",
    index=False)
X_test.to_csv(
    "E:/HealthGuard_AI/data/processed/hypertension_X_test.csv",
    index=False)
y_train.to_csv(
    "E:/HealthGuard_AI/data/processed/hypertension_y_train.csv",
    index=False)
y_test.to_csv(
    "E:/HealthGuard_AI/data/processed/hypertension_y_test.csv",
    index=False)

print("Files Saved:")
print(" hypertension_cleaned_final.csv")
print(" hypertension_X_train.csv")
print(" hypertension_X_test.csv")
print(" hypertension_y_train.csv")
print(" hypertension_y_test.csv")

STEP 9: SAVE CLEANED DATA
Files Saved:
 hypertension_cleaned_final.csv
 hypertension_X_train.csv
 hypertension_X_test.csv
 hypertension_y_train.csv
 hypertension_y_test.csv


In [11]:
# Cleaning Summary

print("=" * 60)
print("   HYPERTENSION CLEANING - SUMMARY REPORT")
print("=" * 60)

print("\nTECHNIQUES USED:")
print("-" * 40)
print("1. Duplicate Removal")
print("2. Missing Value Treatment")
print("3. Target Encoding (Yes=1, No=0)")
print("4. Label Encoding - Categorical Features")
print("5. IQR Outlier Removal")
print("6. Standard Scaling")
print("7. SMOTE")
print("8. Train Test Split 80/20")

print(f"\nFinal Training Set: {len(X_train)}")
print(f"Final Testing Set: {len(X_test)}")

print("\nHypertension Cleaning Complete!")
print("=" * 60)

   HYPERTENSION CLEANING - SUMMARY REPORT

TECHNIQUES USED:
----------------------------------------
1. Duplicate Removal
2. Missing Value Treatment
3. Target Encoding (Yes=1, No=0)
4. Label Encoding - Categorical Features
5. IQR Outlier Removal
6. Standard Scaling
7. SMOTE
8. Train Test Split 80/20

Final Training Set: 1610
Final Testing Set: 389

Hypertension Cleaning Complete!


In [12]:
# Save Scaler for Web App
import pickle

scaler_path = "E:/HealthGuard_AI/models/saved/hypertension_scaler.pkl"
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print(" Hypertension Scaler Saved!")
print(f"Location: {scaler_path}")

 Hypertension Scaler Saved!
Location: E:/HealthGuard_AI/models/saved/hypertension_scaler.pkl
